In [ ]:
# Imports
from pathlib import Path
from deep_events.database.prepare_yaml import prepare_all_folder
from deep_events.csv_to_lines import csv_to_lines
from deep_events.csv_to_gaussian import csv_to_gaussian
from deep_events.gaussians_to_training import main as gaussian_to_training_events
from deep_events.database.construct import reconstruct_from_folder

##### Set the desired parameters

In [ ]:
# The parent data folder(s)
FOLDERS = [Path("my_parent_data_folder")]

# The name of the .csv files with the manual annotations
csv_file_pattern = 'fission_points'

# The original data format
img_types = [r'*.ome.tif*']

# Additional text for the ground truth output files
ground_truth_types = [csv_file_pattern + '_gaussians']

##### From manual annotations to events ready for downstream operations.
First, we generate the *db.yaml* file - containing all necessary metadata for downstream steps - by reading:
- the metadata in the *ome.tiff* file, 
- the optional general metadata contained in *my_parent_data_folder/db_manual.yaml*,
- and the optional specific metadata contained in any *my_data/db_manual.yaml*.

In [ ]:
for folder in FOLDERS:
    prepare_all_folder(Path(folder)) 

##### Generate the *ground-truth* dataset from the manual annotations.
The dataset is a .tiff file, populated only by gaussians in the locations written in the .csv file

In [ ]:
for folder in FOLDERS:
    SIGMA = 10 # the sigma of the gaussian [px]
    csv_to_gaussian(folder, SIGMA, csv_file_pattern)
    
print('\nGround-truth dataset generated.')

##### Generate the event folder - to be used to populate the database

In [ ]:
for gt_type in ground_truth_types:
    settings = {
        'img_identifier': "",
        'gt_identifier': "ground_truth_" + gt_type,  
        'db_name': "db.yaml",
        'channel_contrast': "",
        'label': "",
        'add_post_frames': 0,
        'auto_negatives': 0,
    }
    event_folder = f"event_data_{gt_type}"
    event_folders = [event_folder]

    gaussian_to_training_events(FOLDERS, event_folders, img_types, settings)

print('Event folder generated.')

##### Update the database with the newly generated events, to enable training
Here, we update a MongoDB database with the events later used for training. Please refer to the related section in the README.md file to check how to create it. The database can be later queried to filter training data as desired. The database is deleted (at least for the automatic annotations) every time we run reconstruct_from_folder.

In [ ]:
reconstruct_from_folder(
    "my_parent_data_folder/event_data_fission_points_gaussians", # the event folder just created
    'event_data_fissions_gaussians' # collection name in the database
)